# Signal_maker — WC Testbeam 2026 (Unificato)

Legge i file ROOT in `data/` con naming `sim_<geo>_<src>_X<x>mm_Y<y>mm_Th<theta>deg.root`,  
digitalizza i fotoni in tracce ADC con modello PMT+cavo realistico,  
e salva le waveform in file `.npz` con lo stesso basename del ROOT.

**Configurazione rapida:** modifica `GEO_SEL` e `SRC_SEL` nella cella seguente, poi *Run All*.

In [4]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
import re
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION — modifica qui prima di lanciare
# ============================================================
GEO_SEL  = "single"   # "single" | "triple"
SRC_SEL  = "mu_lngs"     # "e_tb" | "mu_tb" | "mu_lngs" | "gamma" | None (tutti)
MOD_SEL  = None       # (solo triple) "bottom" | "middle" | "top" | None (auto da Y)
RUN_SEL  = None        # None = tutti i file; intero = solo quel run ID


MAX_DEBUG_PLOTS = 0    # numero di waveform da plottare per debug (0 = nessuno)
PLOT_ATTENUATION = True

# ============================================================
# DIGITIZER, PMT & CABLE PARAMETERS
# ============================================================
SAMPLE_RATE_GS    = 2.5
DT_NS             = 1.0 / SAMPLE_RATE_GS   # 0.4 ns/campione
RECORD_LENGTH     = 1024
ADC_BITS          = 12
V_PP_MV           = 1000.0
BASELINE_ADC      = 3800
NOISE_RMS_ADC     = 3.5
TRIGGER_OFFSET_NS = 150.0

# PMT: Hamamatsu H14603-103
G            = 3.0e5
R            = 50.0
e_charge     = 1.602e-19
tr_pmt       = 0.6        # ns rise time
TTS_FWHM_NS  = 0.3
TTS_SIGMA_NS = TTS_FWHM_NS / 2.355
SPE_RES      = 0.40       # sigma/mean risoluzione SPE

# Cavo RG-58
CABLE_LENGTH_M      = 10.0
ATTENUATION_DB      = (15.0 / 100.0) * CABLE_LENGTH_M
TRANSMISSION_FACTOR = 10.0 ** (-ATTENUATION_DB / 20.0)
tr_cable            = 0.14 * CABLE_LENGTH_M
tr_total            = np.sqrt(tr_pmt**2 + tr_cable**2)
sigma_total         = tr_total / 2.22

amp_SPE_mV_ideal      = -(e_charge * G * R) / (sigma_total * 1e-9 * np.sqrt(2 * np.pi)) * 1e3
amp_SPE_mV_attenuated = amp_SPE_mV_ideal * TRANSMISSION_FACTOR

# Finestra di integrazione rapida
QUICK_INT_START_NS = 250.0
QUICK_INT_END_NS   = 410.0

# ============================================================
# HELPER FUNCTIONS
# ============================================================

def mv_to_adc(voltage_mv):
    lsb = V_PP_MV / (2**ADC_BITS)
    adc = BASELINE_ADC + (voltage_mv / lsb)
    adc += np.random.normal(0, NOISE_RMS_ADC, len(adc))
    return np.clip(np.round(adc), 0, (2**ADC_BITS) - 1).astype(np.uint16)

def get_quantum_efficiency(wavelengths_nm):
    """QE curve Hamamatsu H14603-103 (bialkali)."""
    wl = np.array([200., 250., 300., 350., 380., 400., 420., 450., 500., 550., 600., 650., 700.])
    qe = np.array([0.18,  0.25,  0.30,  0.34,  0.35,  0.34,  0.32,  0.28,  0.18,  0.08,  0.03, 0.005, 0.00])
    return np.interp(wavelengths_nm, wl, qe, left=0., right=0.)

def parse_signed(s):
    """Converte 'm200' → -200.0, '200' → 200.0 (prefisso 'm' = negativo)."""
    return -float(s[1:]) if s.startswith('m') else float(s)

# Mappa modulo → (ch_A, ch_B)
# Triple: PMT 0=L_Bot 1=R_Bot 2=L_Mid 3=R_Mid 4=L_Top 5=R_Top
_MOD_PMT = {
    "bottom": (0, 1),
    "middle": (2, 3),
    "top":    (4, 5),
}

def resolve_module(geo, y_mm, mod_sel):
    """
    Restituisce (module_name, ch_A, ch_B).
    - single: sempre "single", (0, 1)
    - triple + mod_sel esplicito: forza il modulo scelto
    - triple + mod_sel=None: auto-detect da Y (soglie ±150 mm)
    """
    if geo == "single":
        return "single", 0, 1

    # Triple con selezione esplicita
    if mod_sel is not None:
        key = mod_sel.lower()
        if key not in _MOD_PMT:
            raise ValueError(f"MOD_SEL non valido: '{mod_sel}'. Valori: bottom | middle | top")
        ch_A, ch_B = _MOD_PMT[key]
        return key, ch_A, ch_B

    # Triple: auto da Y
    if y_mm < -150.0:
        return "bottom", 0, 1
    elif abs(y_mm) <= 150.0:
        return "middle", 2, 3
    else:
        return "top", 4, 5

# Regex per nuovo naming: sim_<geo>_<src>_X<x>mm_Y<y>mm_Th<theta>deg.root
FILE_REGEX = re.compile(
    r'sim_(single|triple)_(e_tb|mu_tb|mu_lngs|gamma)_X(m?\d+)mm_Y(m?\d+)mm_Th(\d+)deg',
    re.IGNORECASE
)

# ============================================================
# FILE DISCOVERY
# ============================================================
input_dir  = "/Users/benussi/Testbeam2026_WC_unified/data"
output_dir = input_dir
os.makedirs(output_dir, exist_ok=True)

src_glob   = "*" if SRC_SEL is None else SRC_SEL
pattern    = f"sim_{GEO_SEL}_{src_glob}_*.root"
root_files = sorted(glob.glob(os.path.join(input_dir, pattern)))

N_PMT = 2 if GEO_SEL == "single" else 6

# Validazione MOD_SEL
if GEO_SEL == "single" and MOD_SEL is not None:
    print(f"[INFO] GEO_SEL='single': MOD_SEL='{MOD_SEL}' ignorato (modulo unico).")
if GEO_SEL == "triple" and MOD_SEL is None:
    print("[INFO] GEO_SEL='triple' + MOD_SEL=None: selezione modulo automatica da Y.")
if GEO_SEL == "triple" and MOD_SEL is not None:
    ch_a_fixed, ch_b_fixed = _MOD_PMT[MOD_SEL.lower()]
    print(f"[INFO] GEO_SEL='triple' + MOD_SEL='{MOD_SEL}': forzo ch_A={ch_a_fixed}, ch_B={ch_b_fixed} per tutti i file.")

if not root_files:
    print(f"[!] Nessun file trovato in {input_dir} con pattern '{pattern}'")
else:
    print(f"Trovati {len(root_files)} file ROOT | GEO={GEO_SEL}  SRC={SRC_SEL}  MOD={MOD_SEL}  N_PMT={N_PMT}")
    print(f" -> Digitizer : {SAMPLE_RATE_GS} GS/s  dt={DT_NS:.3f} ns  {RECORD_LENGTH} campioni = {RECORD_LENGTH*DT_NS:.1f} ns")
    print(f" -> Trigger   : +{TRIGGER_OFFSET_NS} ns  (finestra integrazione {QUICK_INT_START_NS:.0f}–{QUICK_INT_END_NS:.0f} ns)")
    print(f" -> PMT       : G={G:.1e}  TTS={TTS_FWHM_NS} ns FWHM  SPE_res={SPE_RES*100:.0f}%")
    print(f" -> Cavo      : {CABLE_LENGTH_M}m RG-58  -{ATTENUATION_DB:.1f} dB  σ_tot={sigma_total:.2f} ns")
    print(f" -> amp_SPE   : {amp_SPE_mV_attenuated:.2f} mV (attenuato)")

# ============================================================
# MAIN DIGITIZATION LOOP
# ============================================================
time_axis    = np.arange(0, RECORD_LENGTH * DT_NS, DT_NS)   # (N_samp,)
window_start = int(QUICK_INT_START_NS / DT_NS)
window_end   = int(QUICK_INT_END_NS   / DT_NS)
lsb          = V_PP_MV / (2**ADC_BITS)

def digitize_pmt(t_acc):
    """
    Restituisce la waveform ADC per un PMT dato l'array dei tempi di arrivo
    accettati dalla QE. Completamente vettorizzato: nessun loop Python sui fotoni.
    """
    voltage = np.zeros(len(time_axis))
    if len(t_acc) == 0:
        return mv_to_adc(voltage)

    jitters = np.random.normal(0.0, TTS_SIGMA_NS, len(t_acc))
    gains   = np.maximum(0.0, np.random.normal(1.0, SPE_RES, len(t_acc)))
    t_s     = t_acc + TRIGGER_OFFSET_NS + jitters          # (N_ph,)

    # Broadcasting (N_samp, 1) - (1, N_ph) → (N_samp, N_ph)
    dt      = time_axis[:, np.newaxis] - t_s[np.newaxis, :]
    pulses  = np.exp(-0.5 * (dt / sigma_total)**2)         # (N_samp, N_ph)
    voltage = amp_SPE_mV_attenuated * (pulses @ gains)     # (N_samp,)

    return mv_to_adc(voltage)

mock_catalog        = []
events_summary      = []
quick_analysis_data = []
current_run_id      = 1000
debug_plots_saved   = 0

for file_path in root_files:

    if RUN_SEL is not None and current_run_id != RUN_SEL:
        current_run_id += 1
        continue

    basename = os.path.basename(file_path)
    m = FILE_REGEX.search(basename)
    if not m:
        print(f"[SKIP] Nome non riconosciuto: {basename}")
        continue

    geo   = m.group(1)
    src   = m.group(2)
    x_mm  = parse_signed(m.group(3))
    y_mm  = parse_signed(m.group(4))
    theta = float(m.group(5))

    mod, ch_A, ch_B = resolve_module(geo, y_mm, MOD_SEL)

    print(f" -> Run {current_run_id:04d} | {geo} {src} X={x_mm:+.0f}mm Y={y_mm:+.0f}mm Th={theta:.0f}° | mod={mod} (ch_A={ch_A} ch_B={ch_B})")

    # --- Lettura ROOT ---
    try:
        rf = uproot.open(file_path)
        df = rf["Fotoni"].arrays(["EventID", "Arrival_Time_ns", "PMT_ID", "E_Hit_eV"], library="pd")
    except Exception as e:
        print(f"    [!] Errore lettura: {e}")
        events_summary.append({'file': basename, 'valid_events': 'Error'})
        current_run_id += 1
        continue

    if df.empty:
        print("    [!] Nessun fotone registrato.")
        events_summary.append({'file': basename, 'valid_events': 0})
        current_run_id += 1
        continue

    event_ids = np.sort(df.EventID.unique())
    events_summary.append({'file': basename, 'valid_events': len(event_ids)})

    events_data = {}
    qa_list, qb_list = [], []

    for ev_id in event_ids:
        hits_ev    = df[df.EventID == ev_id]
        event_dict = {}
        ph_counts  = {}

        for pmt_id in range(N_PMT):
            hits_pmt = hits_ev[hits_ev.PMT_ID == pmt_id]
            ph_counts[pmt_id] = len(hits_pmt)

            if hits_pmt.empty:
                t_acc = np.array([])
            else:
                wl_nm    = 1240.0 / hits_pmt.E_Hit_eV.values
                accepted = np.random.rand(len(wl_nm)) < get_quantum_efficiency(wl_nm)
                t_acc    = hits_pmt.Arrival_Time_ns.values[accepted]

            event_dict[f'ch_{pmt_id}'] = digitize_pmt(t_acc)

        # Integrazione rapida per analisi attenuazione
        if PLOT_ATTENUATION:
            sA = (event_dict[f'ch_{ch_A}'].astype(float) - BASELINE_ADC) * lsb
            sB = (event_dict[f'ch_{ch_B}'].astype(float) - BASELINE_ADC) * lsb
            qA = -np.sum(sA[window_start:window_end])
            qB = -np.sum(sB[window_start:window_end])
            if qA > 0 and qB > 0:
                qa_list.append(qA)
                qb_list.append(qB)

        # Debug plot waveform
        if debug_plots_saved < MAX_DEBUG_PLOTS:
            nrows = 1 if N_PMT <= 2 else 2
            ncols = N_PMT if N_PMT <= 2 else 3
            fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 4*nrows),
                                     sharex=True, sharey=True, squeeze=False)
            fig.suptitle(f"DEBUG Run {current_run_id} Ev {ev_id} — {basename}", fontsize=11)
            for pid in range(N_PMT):
                ax  = axes[pid // ncols][pid % ncols]
                sig = (event_dict[f'ch_{pid}'].astype(float) - BASELINE_ADC) * lsb
                ax.plot(time_axis, sig, color='darkcyan', lw=1.5)
                ax.axhline(0, color='red', ls='--', alpha=0.6)
                ax.axvline(TRIGGER_OFFSET_NS, color='magenta', ls=':', alpha=0.6)
                ax.axvspan(QUICK_INT_START_NS, QUICK_INT_END_NS, alpha=0.08, color='green')
                if pid in (ch_A, ch_B):
                    for spine in ax.spines.values():
                        spine.set_edgecolor('orange')
                        spine.set_linewidth(2.5)
                ax.set_title(f"PMT {pid} ({ph_counts[pid]} ph)" +
                             (" ★" if pid in (ch_A, ch_B) else ""))
                ax.set_xlim(QUICK_INT_START_NS - 80, QUICK_INT_END_NS + 80)
                ax.grid(True, ls=':', alpha=0.5)
                if pid % ncols == 0: ax.set_ylabel("mV")
                if pid >= N_PMT - ncols: ax.set_xlabel("Time (ns)")
            plt.tight_layout()
            plt.show()
            debug_plots_saved += 1

        events_data[f'run_{current_run_id}_ev_{ev_id}'] = event_dict

    # Analisi rapida per questo file
    if PLOT_ATTENUATION and qa_list:
        qa_arr = np.array(qa_list)
        qb_arr = np.array(qb_list)
        lr_arr = np.log(qa_arr / qb_arr)
        quick_analysis_data.append({
            'file':      basename,
            'geo':       geo,
            'src':       src,
            'X_mm':      x_mm,
            'Y_mm':      y_mm,
            'theta_deg': theta,
            'module':    mod,
            'ch_A':      ch_A,
            'ch_B':      ch_B,
            'Log_Ratio': np.mean(lr_arr),
            'Err_LR':    np.std(lr_arr) / np.sqrt(len(lr_arr)),
            'Q_A':       np.mean(qa_arr),
            'Err_QA':    np.std(qa_arr) / np.sqrt(len(qa_arr)),
            'Q_B':       np.mean(qb_arr),
            'Err_QB':    np.std(qb_arr) / np.sqrt(len(qb_arr)),
            'N_ev':      len(qa_arr),
        })

    # Salva NPZ con stesso basename del ROOT
    npz_name = os.path.splitext(basename)[0] + ".npz"
    np.savez_compressed(os.path.join(output_dir, npz_name), **events_data)
    print(f"    [OK] {npz_name}  ({len(event_ids)} eventi, {len(qa_list)} con segnale)")

    mock_catalog.append({
        'run': current_run_id, 'geo': geo, 'src': src,
        'X_mm': x_mm, 'Y_mm': y_mm, 'theta_deg': theta,
        'module': mod, 'ch_A': ch_A, 'ch_B': ch_B,
        'valid_events': len(event_ids), 'file_npz': npz_name,
    })
    current_run_id += 1

print(f"\n[DONE] Elaborati {current_run_id - 1000} file.")

# ============================================================
# QUICK ANALYSIS PLOTS
# ============================================================
if not PLOT_ATTENUATION or not quick_analysis_data:
    print("Nessun dato per i plot (PLOT_ATTENUATION=False o nessun evento con segnale).")
else:
    df_qa = pd.DataFrame(quick_analysis_data)

    # Un pannello 1×3 per ogni combinazione (Y, theta)
    # Asse X = posizione impatto X in cm
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Quick Analysis — GEO={GEO_SEL}  SRC={SRC_SEL}", fontsize=14)

    groups = df_qa.groupby(['Y_mm', 'theta_deg'])
    for (y_key, th_key), grp in groups:
        grp   = grp.sort_values('X_mm')
        x_cm  = grp['X_mm'].values / 10.0
        lbl   = f"Y={y_key:+.0f}mm  Th={th_key:.0f}°"

        axes[0].errorbar(x_cm, grp['Log_Ratio'], yerr=grp['Err_LR'],  fmt='o-', label=lbl)
        axes[1].errorbar(x_cm, grp['Q_A'],       yerr=grp['Err_QA'], fmt='s-', label=lbl)
        axes[2].errorbar(x_cm, grp['Q_B'],       yerr=grp['Err_QB'], fmt='^-', label=lbl)

    axes[0].set_title(r"$\ln(Q_A\,/\,Q_B)$  [attenuazione]", fontsize=12)
    axes[1].set_title("$Q_A$ — PMT ch_A (sinistra/basso)",    fontsize=12)
    axes[2].set_title("$Q_B$ — PMT ch_B (destra/alto)",       fontsize=12)

    for ax in axes:
        ax.set_xlabel("Posizione fascio X (cm)")
        ax.grid(True, ls=':', alpha=0.7)
        ax.legend(fontsize=7, ncol=2)

    plt.tight_layout()
    plt.show()

    # Tabella riassuntiva
    print(df_qa[['geo','src','X_mm','Y_mm','theta_deg','module','Log_Ratio','Q_A','Q_B','N_ev']]
          .sort_values(['Y_mm','theta_deg','X_mm'])
          .to_string(index=False))

# ============================================================
# SALVA CATALOGHI CSV
# ============================================================
if RUN_SEL is None and mock_catalog:
    cat_path = os.path.join(output_dir, f"catalog_{GEO_SEL}_{SRC_SEL or 'all'}.csv")
    pd.DataFrame(mock_catalog).to_csv(cat_path, index=False)
    print(f"[OK] Catalog → {cat_path}")

    sum_path = os.path.join(output_dir, f"summary_{GEO_SEL}_{SRC_SEL or 'all'}.csv")
    pd.DataFrame(events_summary).to_csv(sum_path, index=False)
    print(f"[OK] Summary → {sum_path}")

Trovati 1 file ROOT | GEO=single  SRC=mu_lngs  MOD=None  N_PMT=2
 -> Digitizer : 2.5 GS/s  dt=0.400 ns  1024 campioni = 409.6 ns
 -> Trigger   : +150.0 ns  (finestra integrazione 250–410 ns)
 -> PMT       : G=3.0e+05  TTS=0.3 ns FWHM  SPE_res=40%
 -> Cavo      : 10.0m RG-58  -1.5 dB  σ_tot=0.69 ns
 -> amp_SPE   : -1.18 mV (attenuato)
 -> Run 1000 | single mu_lngs X=+0mm Y=+0mm Th=0° | mod=single (ch_A=0 ch_B=1)
    [OK] sim_single_mu_lngs_X0mm_Y0mm_Th0deg.npz  (5 eventi, 0 con segnale)

[DONE] Elaborati 1 file.
Nessun dato per i plot (PLOT_ATTENUATION=False o nessun evento con segnale).
[OK] Catalog → /Users/benussi/Testbeam2026_WC_unified/data/catalog_single_mu_lngs.csv
[OK] Summary → /Users/benussi/Testbeam2026_WC_unified/data/summary_single_mu_lngs.csv
